In [ ]:
print("test")

In [ ]:
import pandas as pd
from LabData.DataLoaders.GutMBLoader import GutMBLoader
from LabData.DataLoaders.SubjectLoader import SubjectLoader
from LabData.DataLoaders.DietLoggingLoader import DietLoggingLoader
from LabData.DataLoaders.LifeStyleLoader import LifeStyleLoader
from LabData.DataLoaders.BodyMeasuresLoader import BodyMeasuresLoader
from LabData.DataAnalyses.TenK_Trajectories.utils import get_diet_logging_around_stage
import numpy as np
from matplotlib import cm
from matplotlib.colors import Normalize
import seaborn as sns
import re
import pickle
import matplotlib.pyplot as plt

In [ ]:
stage = 'baseline' # 'baseline' or '02_00_visit' or '04_00_visit'
calc_adherence = True
david = False
show = False

In [ ]:
stage_suf = stage.replace('_00', '') if stage.endswith('_00_visit') else stage

In [ ]:
def explore_columns(df):
    for column in df.columns:
        print(column)
        print(df[column].value_counts())

## study_ids = [10, 1001, 1002]
# study_ids = [10]
study_ids=[10, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010]
subjects_dl = SubjectLoader()
subjects_data = subjects_dl.get_data(groupby_reg='first', study_ids=study_ids)
subjects_df = subjects_data.df

print(subjects_df["age"])

## Load Diet Data

In [ ]:
if stage != 'baseline':
    if not david:
        with open('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/my_lists_diet.pkl', 'rb') as file:
            loaded_lists = pickle.load(file)
        base_features, all_diet_features = loaded_lists
    else:
        with open('/net/mraid20/export/genie/LabData/Analyses/tomerse/david_colab/my_lists_diet.pkl', 'rb') as file:
            loaded_lists = pickle.load(file)
        base_features, all_diet_features = loaded_lists


In [ ]:
with open('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/food_shortnames.pkl', 'rb') as file:
    food_shortnames = pickle.load(file)
food_shortnames

In [ ]:
list(food_shortnames)

In [ ]:
with open('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/nutr_list_aus.pkl', 'rb') as file:
    nutr_list_aus = pickle.load(file)
nutr_list_aus

In [ ]:
dll = DietLoggingLoader()
dlld = dll.get_data(study_ids=study_ids)
log = get_diet_logging_around_stage(dlld.df, stage=stage, delta_before=2, delta_after=14)
# log = get_diet_logging_around_stage(dlld.df, stage=stage, delta_before=10, delta_after=0)
log = log.reset_index()
log = log.set_index(['RegistrationCode','Date','food_id'])
print(log.head(10))

In [ ]:
len(set(log.index.get_level_values(0)))

In [ ]:
len(set(log.index.get_level_values(0).values))

## Add Nutrients

In [ ]:
# nutr_list = list(dll.food_nutrients.columns)
# nutr_list = ['caffeine_mg','calcium_mg','carbohydrate_g',
# 'cholesterol_mg',
# 'energy_kcal',
# 'iron_mg',
# 'magnesium_mg',
# 'niacin_mg',
# 'phosphorus_mg',
# 'potassium_mg',
# 'protein_g',
# 'raevitamina_ug',
# 'riboflavin_mg',
# 'sodium_mg',
# 'thiamin_mg',
# 'totaldietaryfiber_g',
# 'totalfolate_ug',
# 'totallipid_g',
# 'totalmonounsaturatedfattyacids_g',
# 'totalpolyunsaturatedfattyacids_g',
# 'totalsaturatedfattyacids_g',
# 'vitaminb12_ug',
# 'vitaminb6_mg',
# 'vitaminc_mg',
# 'vitamind_iu',
# 'vitamine_mg',
# 'zinc_mg',
# 'alcohol_g']

# # log_date = dll.add_nutrients(log, nutrient_list=nutr_list)
log_date = dll.add_new_nutrients(log)
log_date


# log_date = pd.read_pickle(f'/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/log_date_add_new_nutrients_{stage}.pkl')
# log_date

## Filter Foods Log

In [ ]:
log_date = dll.add_short_food_names(log_date)
log_date = dll.add_food_categories(log_date)
log_date

In [ ]:
log_date = log_date.reset_index()
log_date['Day'] = log_date['Date'].astype(str).str[:10]
log_date['Hour'] = log_date['Date'].astype(str).str[10:16]
log_date.drop('Date', axis=1, inplace=True)
log_date = log_date.set_index(['RegistrationCode','Day','Hour','food_id'])
print(log_date.head())

In [ ]:
# Filter all entries with NaN shortname or Energy
log_date = log_date[~log_date['shortname_eng'].isna()]
log_date = log_date[~log_date['Energy'].isna()]
log_date.shape

In [ ]:
len(set(log_date.index.get_level_values(0)))

In [ ]:
# Filter foods that are not present in >1% of the cohort
log_date = log_date[log_date['shortname_eng'].isin(food_shortnames)]
log_date.shape

In [ ]:
# Filter foods that have no nutrient data, other than sugar substitutes
# Filter rows where all nutrient values are 0
log_date = log_date[~((log_date[nutr_list_aus] == 0).all(axis=1) & (log_date['shortname_eng'] != "Sugar substitute"))]
log_date.shape

In [ ]:
# Filter Poppy seed cake and Salep, which have incorrect nutrient data
log_date = log_date[~log_date['shortname_eng'].isin(["Poppy seed cake", "Salep"])]
# Filter taster's choice which has calories inconsistent with other coffee types
log_date = log_date[~log_date['name'].str.contains("Tasters")]
log_date.shape

In [ ]:
# Filter Halva entries with incorrect nutrient data
log_date = log_date[~((log_date['shortname_eng'] == "Halva") & (log_date['Total lipid (fat)'] == 0))]
log_date.shape

In [ ]:
# Filter unrealistic energy values
log_date = log_date[(log_date["Energy"] >= 0) & (log_date["Energy"] < 3000)]
log_date.shape

In [ ]:
# Filter unrealistic weight values
log_date = log_date[(log_date["weight"] >= 0) & (log_date["weight"] < 2400)]
log_date.shape

In [ ]:
# Filter duplicate entries
log_date_reset = log_date.reset_index()
log_date = log_date_reset[
    ~log_date_reset.duplicated(subset=['RegistrationCode', 'Day', 'food_id', 'Hour', 'weight'], keep='first')
].set_index(['RegistrationCode', 'Day', 'food_id', 'Hour'])
log_date.shape

In [ ]:
# Filter foods without NOVA food scores
NOVA = pd.read_excel('/net/mraid20/export/genie/LabData/Data/10K/foods/processed_foods_mappingV4.xlsx').rename(columns = {'foodid 10k':'food_id'}).dropna(subset=['Score'])
NOVA

In [ ]:
NOVA['food_id'] = NOVA['food_id'].astype(int).astype(str)
log_date_reset = log_date.reset_index()
merged_df = log_date_reset.merge(NOVA[['food_id', 'Score']], on='food_id', how='left')
log_date = merged_df.set_index(log_date.index.names)
log_date = log_date.dropna(subset=['Score'])
log_date = log_date.rename(columns={'Score': 'NOVA_score'})
log_date.shape

In [ ]:
before_any_filters = log_date.reset_index()["RegistrationCode"].nunique()
before_any_filters

In [ ]:
# Filter logging days with <500 calories or >4000 calories

total_energy_per_day = log_date.groupby(['RegistrationCode', 'Day'])['Energy'].transform('sum')
log_date['total_energy_per_day'] = total_energy_per_day
log_date = log_date[(log_date['total_energy_per_day'] >= 500) & (log_date['total_energy_per_day'] <= 4000)]
log_date.shape

In [ ]:
# Validate filtering
log_date.groupby(['RegistrationCode', 'Day'])['Energy'].transform('sum').describe()

In [ ]:
# Filter outlier days that might be under-documentation.

# Assuming `log_date` is your DataFrame
# Calculating energy per day
log_date_energy_per_day = log_date.groupby(['RegistrationCode', 'Day'])['Energy'].sum().reset_index()

# Grouping by RegistrationCode to calculate mean and std
stats = log_date_energy_per_day.groupby('RegistrationCode')['Energy'].agg(['mean', 'std'])

# Merging stats back to a new DataFrame
log_date_with_stats = log_date_energy_per_day.merge(stats, on='RegistrationCode')

# Defining outliers: Energy values outside mean ± 2*std
log_date_with_stats['is_outlier_below'] = (log_date_with_stats['Energy'] < (log_date_with_stats['mean'] - 2.5 * log_date_with_stats['std'])) #| \
                                    #(log_date_with_stats['Energy'] > (log_date_with_stats['mean'] + 2 * log_date_with_stats['std']))

# Filtering outlier days
# outliers = log_date_with_stats[log_date_with_stats['is_outlier']][['RegistrationCode', 'Day', 'Energy', 'is_outlier']]
outliers = log_date_with_stats[log_date_with_stats['is_outlier_below']][['RegistrationCode', 'Day', 'Energy', 'is_outlier_below']]

# Display the outliers
print(outliers)
# print(log_date_with_stats[log_date_with_stats['RegistrationCode'] == 'EXAMPLE_ID'])


In [ ]:
# Filter outlier days
log_date = log_date[~log_date_with_stats.set_index(['RegistrationCode', 'Day'])['is_outlier_below']]
log_date.shape

In [ ]:
log_date.head(20)

In [ ]:
# Filter People with less than 3 days of diet documentation.
# Group by RegistrationCode and count the number of unique days using the index level 'Day'
unique_day_counts = log_date.groupby('RegistrationCode').apply(lambda x: x.index.get_level_values('Day').nunique())

# Filter out RegistrationCodes with less than 3 unique days
valid_registration_codes = unique_day_counts[unique_day_counts >= 3].index

# Create a new DataFrame with only the valid RegistrationCodes
log_date = log_date.loc[valid_registration_codes]

# Display the filtered DataFrame
print(log_date.shape)


In [ ]:
people_more_than_3 = log_date.reset_index()["RegistrationCode"].nunique()
people_more_than_3

In [ ]:
people_more_than_8 = log_date.reset_index()["RegistrationCode"].nunique()
people_more_than_8

In [ ]:
before_any_filters

In [ ]:
unique_day_counts

In [ ]:
plt.hist(unique_day_counts, bins=range(1, unique_day_counts.max()+2))
plt.xticks(range(1, unique_day_counts.max()+1))
plt.show()


In [ ]:
exact_7_days = (unique_day_counts == 7).sum()
more_than_14_days = (unique_day_counts >= 14).sum()
less_than_14_days = ((unique_day_counts < 14) & (unique_day_counts > 7)).sum()

print(f"Number of unique_day_counts with exactly 7 days: {exact_7_days}")
print(f"Number of unique_day_counts with 14 or more days: {more_than_14_days}")
print(f"Number of unique_day_counts with less than 14 days: {less_than_14_days}")


In [ ]:
days_saved = ((unique_day_counts < 8) & (unique_day_counts >= 3)).sum()

print(f"Number of unique_day_counts with 4<=x<=7 days: {days_saved}")


In [ ]:
# validate filtering:
# Group by RegistrationCode and count the number of unique days using the index level 'Day'
unique_day_counts_after = log_date.groupby('RegistrationCode').apply(lambda x: x.index.get_level_values('Day').nunique())
plt.hist(unique_day_counts_after, bins=range(1, unique_day_counts_after.max()+2))
plt.xticks(range(1, unique_day_counts_after.max()+1))
plt.show()

In [ ]:
# # Validate filtering:
# # Group by RegistrationCode and count the number of unique days using the 'Date' index level
# unique_day_counts_after = log_date.groupby('RegistrationCode').apply(
#     lambda x: pd.to_datetime(x.index.get_level_values('Date')).normalize().nunique()
# )

# # Plot histogram of the unique day counts
# plt.hist(unique_day_counts_after, bins=range(1, unique_day_counts_after.max() + 2))
# plt.xticks(range(1, unique_day_counts_after.max() + 1))
# plt.show()


In [ ]:
log_date.groupby(['RegistrationCode', 'Day'])['Energy'].sum().describe()
# max_calories_registration_code = log_date.groupby(['RegistrationCode', 'Day'])['Energy'].sum().idxmax()[0]
# print(max_calories_registration_code)


In [ ]:
log_date.reset_index()["RegistrationCode"].nunique()

In [ ]:
# Log day grouping
log_day = log_date.groupby(['RegistrationCode', 'Day'])[nutr_list_aus].sum()
log_day = log_day[log_day['Energy']>500]
log_day = log_day[log_day['Energy']<4000]
log_day

In [ ]:
# log grouped grouping
log_reset = log_day.reset_index().drop("Day", axis=1)

# Group by RegistrationCode and calculate mean for all numeric columns
log_grouped = log_reset.groupby('RegistrationCode').mean()

In [ ]:
log_grouped

In [ ]:
# Compare scales to aus data, do I need to normalize?
diet_aus = pd.read_csv('/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/nastya/diet_predictions/nutr_full_clean_aus.csv', index_col=0)
diet_aus

In [ ]:
# Relative nutrients to energy
nutr_list_no_energy = [nutrient for nutrient in nutr_list_aus if nutrient != "Energy"]
log_grouped[nutr_list_no_energy] = log_grouped[nutr_list_no_energy].div(log_grouped['Energy'], axis=0)

In [ ]:
home_path = '/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/'
diet_mb_10k = pd.read_pickle(home_path + f"data/segal_species/diet_mb.pkl")
with open(home_path + f'data/segal_species/my_lists.pkl', 'rb') as file:
    loaded_lists = pickle.load(file)
base_features_10k, all_features_10k, targets_10k = loaded_lists


In [ ]:
common_index = log_grouped.index.intersection(diet_mb_10k.index)
filtered = log_grouped.loc[common_index]
filtered

In [ ]:
def explore_columns(df):
    for column in df.columns:
        print(column)
        print(df[column].value_counts())

## study_ids = [10, 1001, 1002]
# study_ids = [10]
study_ids=[10, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010]
subjects_dl = SubjectLoader()
subjects_data = subjects_dl.get_data(groupby_reg='first', study_ids=study_ids)
subjects_df = subjects_data.df

print(subjects_df["age"])

In [ ]:
base_features = ["age", "gender"]
diet_mb = filtered.join(subjects_df[base_features])
diet_mb = diet_mb.reset_index(level=[1], drop=True)
diet_mb = diet_mb.dropna()
diet_mb = diet_mb.rename(columns={"gender": "sex"})
diet_mb

In [ ]:
diet_mb = diet_mb.join(diet_mb_10k[targets_10k])
diet_mb

In [ ]:
diet_mb = diet_mb.join(diet_mb_10k[["Richness", "Shannon_diversity", "Faith_index", "bmi"]])
diet_mb

In [ ]:
diet_mb.columns

In [ ]:
diet_mb.to_pickle(home_path + f"data/segal_species/diet_mb_new_nutrients.pkl")